# 第六章：基于 Diff 的进化

**系列：** OpenEvolve — 从零到专家·手撕全流程  
**上一章：** [05 — 级联评估](./05-cascade-evaluation.ipynb)  
**本章内容：** 为什么全量重写效率低、SEARCH/REPLACE 差异格式、diff 解析与应用、以及差异进化的完整流程。

---

## 1. 痛点：全量重写太浪费了

在第四章中，LLM 每次变异都输出**完整的新代码**。想象一下：

```
原始代码：100 行
LLM 需要修改：第 42 行的一个 bug

全量重写模式：
  → LLM 输出 100 行代码（其中 99 行和原来一样）
  → 浪费 Token：~99%
  → 风险：LLM 可能不小心改了其他行

Diff 模式：
  → LLM 只输出改动的 1 行
  → 节约 Token：~99%
  → 精准：不会误伤其他代码
```

**生活类比：** 修改一篇论文时，导师不会把整篇论文重写一遍，而是用红笔标出修改的地方。  
SEARCH/REPLACE 就是 LLM 的「红笔批注」。

## 2. 理论：SEARCH/REPLACE 格式

### 2.1 格式定义

OpenEvolve 使用类似 Git 合并冲突的标记来表示代码差异：

```
<<<<<<< SEARCH
[要查找的原始代码 — 必须精确匹配]
=======
[替换成的新代码]
>>>>>>> REPLACE
```

**关键规则：**
1. SEARCH 部分必须**逐行精确匹配**原始代码（包括空格和缩进）
2. 一次响应中可以包含**多个** SEARCH/REPLACE 块
3. 多个块**按顺序**依次应用（后面的块在前面的结果上操作）
4. 如果 SEARCH 找不到匹配，该块被**静默跳过**

### 2.2 为什么选择这个格式？

| 候选格式 | 优点 | 缺点 | 适合 LLM？ |
|---------|------|------|-----------|
| unified diff (`+/-`) | 标准格式 | LLM 容易搞错行号 | 一般 |
| SEARCH/REPLACE | 直觉、无行号 | 需要精确匹配 | 很好 |
| JSON patch | 结构化 | 语法复杂 | 差 |
| 行号替换 | 精确 | LLM 容易数错行 | 差 |

SEARCH/REPLACE 的设计核心：**让 LLM 用人类直觉来描述改动**——「把这段改成那段」。

### 2.3 正则表达式

提取 SEARCH/REPLACE 块的正则：

```python
pattern = r"<<<<<<< SEARCH\n(.*?)=======\n(.*?)>>>>>>> REPLACE"
```

- `(.*?)` — 非贪婪匹配，捕获 SEARCH 和 REPLACE 的内容
- 使用 `re.DOTALL` 标志，让 `.` 匹配换行符（支持多行代码）

## 3. 从零实现：Diff 解析器

第一步：从 LLM 的响应文本中**提取**所有 SEARCH/REPLACE 块。

In [ ]:
import re
from typing import List, Tuple, Optional


# OpenEvolve 默认的 diff 正则模式
DEFAULT_DIFF_PATTERN = r"<<<<<<< SEARCH\n(.*?)=======\n(.*?)>>>>>>> REPLACE"


def extract_diffs(
    diff_text: str,
    pattern: str = DEFAULT_DIFF_PATTERN,
) -> List[Tuple[str, str]]:
    """从 LLM 响应中提取所有 SEARCH/REPLACE 块
    
    Args:
        diff_text: LLM 的完整响应文本
        pattern: 正则表达式模式
    
    Returns:
        [(search_text, replace_text), ...] 列表
    """
    blocks = re.findall(pattern, diff_text, re.DOTALL)
    # 去掉尾部空白，但保留行内缩进
    return [(s.rstrip(), r.rstrip()) for s, r in blocks]


# 测试
llm_response = '''我分析了代码，建议做两处修改：

第一处：修复排序方向
<<<<<<< SEARCH
    if arr[j] > arr[j+1]:
=======
    if arr[j] < arr[j+1]:  # 改为降序
>>>>>>> REPLACE

第二处：添加边界检查
<<<<<<< SEARCH
def sort_func(arr):
=======
def sort_func(arr):
    if len(arr) <= 1:
        return arr
>>>>>>> REPLACE
'''

diffs = extract_diffs(llm_response)
print(f"提取到 {len(diffs)} 个 diff 块：")
for i, (search, replace) in enumerate(diffs):
    print(f"\n--- Diff {i+1} ---")
    print(f"SEARCH:  {repr(search)}")
    print(f"REPLACE: {repr(replace)}")

## 4. 从零实现：Diff 应用器

核心算法：逐行匹配 + 替换。

```
原始代码（按行分割）:
  Line 0: def sort_func(arr):
  Line 1:     arr = arr[:]
  Line 2:     for i in range(len(arr)):
  Line 3:         for j in range(len(arr)-i-1):
  Line 4:             if arr[j] > arr[j+1]:          ← SEARCH 匹配到这里
  Line 5:                 arr[j], arr[j+1] = ...
  Line 6:     return arr

应用 diff:
  → 找到 Line 4 匹配 SEARCH
  → 用 REPLACE 内容替换 Line 4
  → 其他行不变
```

In [ ]:
def apply_diff(
    original_code: str,
    diff_text: str,
    pattern: str = DEFAULT_DIFF_PATTERN,
) -> str:
    """将 SEARCH/REPLACE diff 应用到原始代码
    
    算法：
    1. 提取所有 diff 块
    2. 对每个块，在代码中找到 SEARCH 的精确匹配位置
    3. 用 REPLACE 替换该位置
    4. 多个块按顺序依次应用
    """
    result_lines = original_code.split('\n')
    diff_blocks = extract_diffs(diff_text, pattern)
    
    applied_count = 0
    
    for search_text, replace_text in diff_blocks:
        search_lines = search_text.split('\n')
        replace_lines = replace_text.split('\n')
        
        # 在 result_lines 中找到精确匹配
        found = False
        for i in range(len(result_lines) - len(search_lines) + 1):
            if result_lines[i:i+len(search_lines)] == search_lines:
                # 替换！
                result_lines[i:i+len(search_lines)] = replace_lines
                applied_count += 1
                found = True
                break  # 只替换第一处匹配
        
        if not found:
            print(f"  [Warning] SEARCH not found: {repr(search_text[:50])}...")
    
    print(f"  Applied {applied_count}/{len(diff_blocks)} diffs")
    return '\n'.join(result_lines)


# 演示
original = '''def sort_func(arr):
    arr = arr[:]
    for i in range(len(arr)):
        for j in range(0, len(arr)-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr'''

diff = '''让排序更高效：
<<<<<<< SEARCH
    for i in range(len(arr)):
        for j in range(0, len(arr)-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
=======
    # 优化：添加提前终止
    for i in range(len(arr)):
        swapped = False
        for j in range(0, len(arr)-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
                swapped = True
        if not swapped:
            break
>>>>>>> REPLACE
'''

result = apply_diff(original, diff)
print("\n=== 应用 diff 后的代码 ===")
print(result)

## 5. 多个 Diff 的顺序应用

LLM 可以在一次响应中建议多处修改。关键点：
- 每个 diff 应用在**前一个 diff 的结果**上（不是原始代码上）
- 顺序很重要：如果 diff A 改了某行，diff B 的 SEARCH 要匹配改后的版本

**数值例子：**
> 原始代码有 10 行，Diff 1 在第 3 行插入 2 行（变成 12 行），  
> Diff 2 搜索第 7 行——但此时第 7 行已经变了（原来的第 5 行变成了第 7 行）。  
> 所以 Diff 2 的 SEARCH 必须匹配**当前状态**的第 7 行，而非原始的第 7 行。

In [ ]:
# 演示：两个 diff 依次应用
original2 = '''x = 1
y = 2
z = x + y
print(z)'''

multi_diff = '''
<<<<<<< SEARCH
x = 1
=======
x = 10
>>>>>>> REPLACE

<<<<<<< SEARCH
y = 2
=======
y = 20
>>>>>>> REPLACE
'''

result2 = apply_diff(original2, multi_diff)
print("\n=== 结果 ===")
print(result2)

# 验证：执行结果应该是 z = 30
exec_ns = {}
exec(result2, exec_ns)
print(f"\n执行结果：z = {exec_ns.get('z', 'undefined')}")

## 6. 全量重写模式解析器

并非所有场景都适合 diff 模式。对于**从零创建**的代码，LLM 需要输出完整程序。

全量重写模式下，LLM 的响应格式：
````
```python
def sort_func(arr):
    ...
```
````

| 模式 | 适合场景 | 不适合场景 |
|------|---------|----------|
| Diff | 已有代码的渐进改进 | 从零创建新代码 |
| 全量重写 | 创建新代码、小程序 | 大程序（浪费 Token） |

In [ ]:
def parse_full_rewrite(llm_response: str, language: str = 'python') -> Optional[str]:
    """从 LLM 响应中提取完整的代码块
    
    尝试顺序：
    1. 特定语言的代码块: ```python ... ```
    2. 通用代码块: ``` ... ```
    3. 整个响应文本（回退）
    """
    # 尝试 1: ```python ... ```
    pattern = rf'```{language}\s*\n(.*?)```'
    match = re.search(pattern, llm_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # 尝试 2: ``` ... ```
    pattern = r'```\s*\n(.*?)```'
    match = re.search(pattern, llm_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # 回退: 整个文本
    return llm_response.strip() if llm_response.strip() else None


# 测试
response1 = '''这是改进后的排序算法：

```python
def sort_func(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    left = [x for x in arr[1:] if x <= pivot]
    right = [x for x in arr[1:] if x > pivot]
    return sort_func(left) + [pivot] + sort_func(right)
```

这个实现使用了快速排序。'''

code1 = parse_full_rewrite(response1)
print("=== 提取的代码 ===")
print(code1)

# 验证可运行
ns = {}
exec(code1, ns)
print(f"\n排序 [5,3,8,1]: {ns['sort_func']([5,3,8,1])}")

## 7. Diff 摘要：记录改动历史

每次变异后，OpenEvolve 会生成一个**人类可读的改动摘要**，存入程序的元数据中。
这些摘要在后续的 prompt 中展示给 LLM，让它知道「之前都做了哪些修改」。

摘要格式分两种：
- **单行改动**：`Change 1: 'x = 1' to 'x = 2'`
- **多行改动**：展示 SEARCH/REPLACE 内容（截断过长的行）

In [ ]:
def format_diff_summary(
    diff_blocks: List[Tuple[str, str]],
    max_line_len: int = 80,
    max_lines: int = 20,
) -> str:
    """生成人类可读的 diff 摘要"""
    if not diff_blocks:
        return 'No changes'
    
    parts = []
    for i, (search, replace) in enumerate(diff_blocks):
        search_lines = search.split('\n')
        replace_lines = replace.split('\n')
        
        # 单行改动：紧凑格式
        if len(search_lines) == 1 and len(replace_lines) == 1:
            s = search[:max_line_len] + ('...' if len(search) > max_line_len else '')
            r = replace[:max_line_len] + ('...' if len(replace) > max_line_len else '')
            parts.append(f"Change {i+1}: '{s}' -> '{r}'")
        else:
            # 多行改动：块格式
            def truncate_block(lines, max_l):
                result = []
                for line in lines[:max_l]:
                    if len(line) > max_line_len:
                        result.append(line[:max_line_len] + '...')
                    else:
                        result.append(line)
                if len(lines) > max_l:
                    result.append(f'... ({len(lines) - max_l} more lines)')
                return result
            
            s_block = '\n'.join(truncate_block(search_lines, max_lines))
            r_block = '\n'.join(truncate_block(replace_lines, max_lines))
            parts.append(
                f"Change {i+1}:\n"
                f"  SEARCH ({len(search_lines)} lines):\n"
                f"    {s_block}\n"
                f"  REPLACE ({len(replace_lines)} lines):\n"
                f"    {r_block}"
            )
    
    return '\n'.join(parts)


# 演示
test_diffs = [
    ('x = 1', 'x = 10'),  # 单行
    ('for i in range(n):\n    print(i)', 'for i in range(n):\n    if i % 2 == 0:\n        print(i)'),  # 多行
]

summary = format_diff_summary(test_diffs)
print("=== Diff 摘要 ===")
print(summary)

## 8. 完整演示：Diff 模式 vs 全量重写

现在对比两种模式在进化循环中的效果。

模拟场景：有一个带 bug 的排序函数，用 Mock LLM 分别在两种模式下修复它。

In [ ]:
import time as time_module

# 带 bug 的原始代码
buggy_code = '''def sort_func(arr):
    arr = arr[:]
    n = len(arr)
    for i in range(n):
        for j in range(0, n-1):  # Bug: 应该是 n-i-1
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr'''


# 模拟 LLM 的 diff 模式响应
def mock_llm_diff_response(code: str) -> str:
    return '''修复排序边界 bug：
<<<<<<< SEARCH
        for j in range(0, n-1):  # Bug: 应该是 n-i-1
=======
        for j in range(0, n-i-1):
>>>>>>> REPLACE
'''


# 模拟 LLM 的全量重写响应
def mock_llm_rewrite_response(code: str) -> str:
    return '''```python
def sort_func(arr):
    arr = arr[:]
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr
```'''


# ===== 对比 =====
print("=" * 60)
print("对比：Diff 模式 vs 全量重写")
print("=" * 60)

# Diff 模式
diff_response = mock_llm_diff_response(buggy_code)
diff_result = apply_diff(buggy_code, diff_response)
diff_tokens = len(diff_response)  # 粗略估算

# 全量重写
rewrite_response = mock_llm_rewrite_response(buggy_code)
rewrite_result = parse_full_rewrite(rewrite_response)
rewrite_tokens = len(rewrite_response)

# 验证两者结果相同
ns1, ns2 = {}, {}
exec(diff_result, ns1)
exec(rewrite_result, ns2)
test_input = [5, 3, 8, 1, 9, 2]

print(f"\n原始代码排序 [5,3,8,1,9,2]:")
ns0 = {}
exec(buggy_code, ns0)
print(f"  Buggy: {ns0['sort_func'](test_input[:])}")

print(f"\nDiff 模式结果: {ns1['sort_func'](test_input[:])}")
print(f"全量重写结果: {ns2['sort_func'](test_input[:])}")
print(f"\n--- Token 消耗对比 ---")
print(f"Diff 响应长度:   {diff_tokens:>6} 字符")
print(f"重写响应长度:   {rewrite_tokens:>6} 字符")
savings = (1 - diff_tokens / rewrite_tokens) * 100
print(f"Diff 节省:      {savings:.0f}%")

## 9. 基于 Diff 的进化循环

将 diff 解析整合到完整的进化循环中。关键变化：
- 变异函数返回 diff 文本（而非完整代码）
- 评估前先 `apply_diff` 将 diff 应用到父代码
- 保存 diff 摘要到程序元数据中

In [ ]:
import random


def diff_based_evolution(
    initial_code: str,
    diff_mutator,  # 函数：code -> diff_response
    evaluator_fn,  # 函数：code -> score
    generations: int = 15,
):
    """基于 Diff 的进化循环"""
    best_code = initial_code
    best_score = evaluator_fn(initial_code)
    
    history = {
        'scores': [best_score],
        'diffs_applied': 0,
        'diffs_failed': 0,
        'summaries': [],
    }
    
    for gen in range(generations):
        # 1. LLM 生成 diff
        diff_response = diff_mutator(best_code)
        
        # 2. 提取 diff 块
        diff_blocks = extract_diffs(diff_response)
        
        if not diff_blocks:
            history['diffs_failed'] += 1
            history['scores'].append(best_score)
            continue
        
        # 3. 应用 diff
        child_code = apply_diff(best_code, diff_response)
        
        # 4. 生成摘要
        summary = format_diff_summary(diff_blocks)
        history['summaries'].append(summary)
        
        # 5. 评估
        try:
            child_score = evaluator_fn(child_code)
        except Exception:
            child_score = 0.0
        
        # 6. 选择
        if child_score > best_score:
            best_code = child_code
            best_score = child_score
            history['diffs_applied'] += 1
        
        history['scores'].append(best_score)
    
    return best_code, best_score, history


# 简单评估函数：排序正确性
def eval_sort(code: str) -> float:
    try:
        ns = {}
        exec(code, ns)
        f = ns['sort_func']
        tests = [
            ([5,3,8,1], [1,3,5,8]),
            ([], []),
            ([1], [1]),
            ([3,2,1], [1,2,3]),
            ([-1,3,-2], [-2,-1,3]),
        ]
        passed = sum(1 for inp, exp in tests if f(inp[:]) == exp)
        return passed / len(tests)
    except Exception:
        return 0.0


# Mock diff mutator：模拟 LLM 生成不同的 diff
def mock_diff_mutator(code: str) -> str:
    mutations = [
        # 修复 bug
        '<<<<<<< SEARCH\n        for j in range(0, n-1):  # Bug: 应该是 n-i-1\n=======\n        for j in range(0, n-i-1):\n>>>>>>> REPLACE',
        # 添加提前终止优化
        '<<<<<<< SEARCH\n    for i in range(n):\n=======\n    for i in range(n):\n        swapped = False\n>>>>>>> REPLACE',
        # 无效修改（SEARCH 找不到匹配）
        '<<<<<<< SEARCH\nthis_line_doesnt_exist\n=======\nreplacement\n>>>>>>> REPLACE',
        # 空响应
        'I think the code looks good as is.',
    ]
    return random.choice(mutations)


random.seed(123)
final_code, final_score, hist = diff_based_evolution(
    initial_code=buggy_code,
    diff_mutator=mock_diff_mutator,
    evaluator_fn=eval_sort,
    generations=15,
)

print(f"\n最终得分: {final_score:.2f}")
print(f"成功应用的 diff: {hist['diffs_applied']}")
print(f"失败的 diff: {hist['diffs_failed']}")
print(f"\n最终代码:")
print(final_code)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：适应度曲线
ax1.plot(hist['scores'], 'b-o', linewidth=2, markersize=4)
ax1.set_xlabel('Generation', fontsize=12)
ax1.set_ylabel('Fitness', fontsize=12)
ax1.set_title('Diff-Based Evolution Progress', fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.05, 1.05)

# 右图：Token 消耗对比
# 假设原始代码 200 字符，diff 平均 80 字符
code_len = 200
diff_len = 80
gens = 30
rewrite_total = code_len * gens
diff_total = diff_len * gens

ax2.bar(['Full Rewrite\n(30 gens)', 'Diff Mode\n(30 gens)'],
        [rewrite_total, diff_total],
        color=['#e74c3c', '#2ecc71'])
ax2.set_ylabel('Total Output Tokens (chars)', fontsize=12)
ax2.set_title('Token Consumption Comparison', fontsize=14)
savings = (1 - diff_total/rewrite_total) * 100
ax2.text(1, diff_total + 200, f'Saved {savings:.0f}%',
         ha='center', fontsize=14, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

## 10. 我们的实现 vs. OpenEvolve 源码

| 概念 | 我们的实现 | OpenEvolve 源码 | 差异说明 |
|------|-----------|----------------|----------|
| Diff 提取 | `extract_diffs()` | `code_utils.py:L78-92` | 逻辑一致 |
| Diff 应用 | `apply_diff()` | `code_utils.py:L40-75` | 逻辑一致 |
| 全量重写解析 | `parse_full_rewrite()` | `code_utils.py:L95-120` | 逻辑一致 |
| Diff 摘要 | `format_diff_summary()` | `code_utils.py:L136-166` | 我们简化了截断逻辑 |
| 模式选择 | 手动切换 | `config.diff_based_evolution` | 源码通过配置切换 |
| 正则模式 | 硬编码 | `config.diff_pattern` (可配置) | 源码允许自定义 |
| 多文件 diff | 未实现 | `split_diffs_by_target()` | 源码支持多目标文件 |

### 关键配置项

```yaml
# 启用 diff 模式（推荐用于大代码库）
diff_based_evolution: true

# 代码最大长度限制
max_code_length: 10000

# 自定义 diff 正则（高级用法）
diff_pattern: "<<<<<<< SEARCH\\n(.*?)=======\\n(.*?)>>>>>>> REPLACE"

# Diff 摘要配置
prompt:
  diff_summary_max_line_len: 100
  diff_summary_max_lines: 30
```

### 模式选择指南

| 场景 | 推荐模式 | 原因 |
|------|---------|------|
| 算法优化（已有代码） | Diff | 精准修改，省 Token |
| 从零生成新代码 | 全量重写 | 没有现有代码可 diff |
| 小程序（< 50 行） | 全量重写 | diff 优势不大 |
| 大程序（> 200 行） | Diff | Token 节约显著 |

### 关键源码位置

| 功能 | 文件 | 行号 |
|------|------|------|
| diff 正则定义 | `config.py` | L423 |
| diff 提取 | `code_utils.py` | L78-92 |
| diff 应用 | `code_utils.py` | L40-75 |
| 全量重写解析 | `code_utils.py` | L95-120 |
| diff 摘要格式化 | `code_utils.py` | L136-166 |
| 多文件路由 | `code_utils.py` | L263-299 |
| 模板选择逻辑 | `sampler.py` | L88-100 |
| diff 模板 | `templates.py` | L23-66 |
| 迭代中的 diff 处理 | `iteration.py` | L98-144 |

## 11. 本章总结

```mermaid
flowchart LR
    subgraph "Diff 模式"
        A[父代码] --> B[PromptBuilder]
        B --> C[LLM]
        C -->|SEARCH/REPLACE| D[extract_diffs]
        D --> E[apply_diff]
        E --> F[子代码]
    end
    subgraph "全量重写模式"
        G[父代码] --> H[PromptBuilder]
        H --> I[LLM]
        I -->|完整代码| J[parse_full_rewrite]
        J --> K[子代码]
    end
```

### 核心收获

| 概念 | 一句话总结 |
|------|----------|
| SEARCH/REPLACE | LLM 的「红笔批注」——只描述改动部分 |
| 精确匹配 | SEARCH 必须逐行精确匹配，这是安全性保证 |
| 顺序应用 | 多个 diff 依次应用在前一个的结果上 |
| 模式选择 | 大代码用 Diff，小代码/新代码用全量重写 |

### 下一章预告

到目前为止，我们的进化循环是**串行**的——每次只评估一个程序。  
当评估耗时较长时（比如运行复杂的测试套件），这成了瓶颈。  
下一章我们实现 **进程并行** —— 同时评估多个程序，充分利用多核 CPU。

## 12. 保存模块

将本章实现的 diff 工具保存到 `our-implementation/` 目录。

In [ ]:
save_code = '''
"""Diff 工具 - 第六章实现

SEARCH/REPLACE 差异格式的解析与应用。
"""
import re
from typing import List, Tuple, Optional

DEFAULT_DIFF_PATTERN = r"<<<<<<< SEARCH\\n(.*?)=======\\n(.*?)>>>>>>> REPLACE"


def extract_diffs(diff_text, pattern=DEFAULT_DIFF_PATTERN):
    blocks = re.findall(pattern, diff_text, re.DOTALL)
    return [(s.rstrip(), r.rstrip()) for s, r in blocks]


def apply_diff(original_code, diff_text, pattern=DEFAULT_DIFF_PATTERN):
    result_lines = original_code.split("\\n")
    diff_blocks = extract_diffs(diff_text, pattern)
    for search_text, replace_text in diff_blocks:
        search_lines = search_text.split("\\n")
        replace_lines = replace_text.split("\\n")
        for i in range(len(result_lines) - len(search_lines) + 1):
            if result_lines[i:i+len(search_lines)] == search_lines:
                result_lines[i:i+len(search_lines)] = replace_lines
                break
    return "\\n".join(result_lines)


def parse_full_rewrite(llm_response, language="python"):
    pattern = rf"```{language}\\s*\\n(.*?)```"
    match = re.search(pattern, llm_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    pattern = r"```\\s*\\n(.*?)```"
    match = re.search(pattern, llm_response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return llm_response.strip() if llm_response.strip() else None


def format_diff_summary(diff_blocks, max_line_len=80, max_lines=20):
    if not diff_blocks:
        return "No changes"
    parts = []
    for i, (search, replace) in enumerate(diff_blocks):
        s_lines = search.split("\\n")
        r_lines = replace.split("\\n")
        if len(s_lines) == 1 and len(r_lines) == 1:
            parts.append(f"Change {i+1}: \'{search}\' -> \'{replace}\'")
        else:
            parts.append(f"Change {i+1}: {len(s_lines)} lines -> {len(r_lines)} lines")
    return "\\n".join(parts)
'''

import os
os.makedirs('../our-implementation', exist_ok=True)
with open('../our-implementation/diff_utils.py', 'w', encoding='utf-8') as f:
    f.write(save_code)

print('已保存到 our-implementation/diff_utils.py ✓')